# Step 0: Importing the necessary libraries

In [1]:
# For file and system operations
from urllib import request
from pathlib import Path
from __future__ import annotations
import os
import yaml
import hashlib
import json
import tempfile
import shutil

# For data manipulation and analysis
import pandas as pd

# For numerical operations
import numpy as np 
# For type hinting
from typing import Dict, Optional

# For date and time handling
from datetime import datetime

from dataclasses import dataclass, field
from typing import Iterable, Dict, Optional, List

# For plotting
import matplotlib.pyplot as plt

#  For linear regression
from scipy.stats import linregress

# Attempt to import pycountry for geographic conversions
try:
    import pycountry_convert as pc
except ImportError:
    print("Warning: pycountry_convert not found. Geography mapping will be limited.")
    pc = None

# For suppressing warnings
import warnings
warnings.filterwarnings('ignore')



# Step 1: Load the data

## 1. Geography Mapping Logic
Handles the translation of ISO codes into continents and subregions. It uses a configuration file to define custom subregions (e.g., "Eastern Africa").

In [2]:
# 1️⃣ Set dataset path
iso_dataset_path = Path("dataset/iso_metadata.csv")

# 2️⃣ Download file if it doesn't exist
if not iso_dataset_path.exists():
    print("Downloading iso_metadata.csv...")
    iso_dataset_path.parent.mkdir(parents=True, exist_ok=True)  
    url = "..."
    try:
        response = request.urlopen(url)
        with open(iso_dataset_path, 'w', encoding='utf-8') as f:
            f.write(response.read().decode('utf-8'))
        print("✅ Download complete.")
    except request.URLError as e:
        print(f"❌ Error downloading dataset: {e}")
else:
    print("✅ Dataset already exists. Loading from file.")

# 3️⃣ Load dataset into pandas DataFrame
# Using comma as separator (default for this file)
iso_dataset_items = pd.read_csv(iso_dataset_path)

# 4️⃣ Inspect the first and last 5 rows
print("First 5 rows:")
print(iso_dataset_items.head())

print("\nLast 5 rows:")
iso_dataset_items.tail()

✅ Dataset already exists. Loading from file.
First 5 rows:
                    name  iso
0            Afghanistan  AFG
1  Akrotiri and Dhekelia  XAD
2                  Åland  ALA
3                Albania  ALB
4                Algeria  DZA

Last 5 rows:


,name,iso
242,"Virgin Islands, U.S.",VIR
243,Wallis and Futuna,WLF
244,Yemen,YEM
245,Zambia,ZMB
246,Zimbabwe,ZWE


## Apply the continent mapping

In [3]:
# Continent code → continent name mapping

CONTINENT_CODE_TO_NAME = {
    "AF": "Africa",
    "AS": "Asia",
    "EU": "Europe",
    "NA": "North America",
    "SA": "South America",
    "OC": "Oceania",
    "AN": "Antarctica",
}

# ISO3 codes that don't have standard mappings
SPECIAL_ISO3_TO_ISO2 = {"XAD", "XCA", "XCL", "XKO", "XPI", "XSP", "XSX"}

def iso3_to_continent(iso3: str) -> str:
    """
    Converts a 3-letter ISO country code to a continent name.
    Returns "Unknown" if mapping fails.
    """
    if not iso3 or pc is None:
        return "Unknown"

    iso3_clean = iso3.upper().strip()
    if iso3_clean in SPECIAL_ISO3_TO_ISO2:
        return "Unknown"

    try:
        iso2 = pc.country_alpha3_to_country_alpha2(iso3_clean)
        continent_code = pc.country_alpha2_to_continent_code(iso2)
        return CONTINENT_CODE_TO_NAME.get(continent_code, "Unknown")
    except Exception:
        return "Unknown"


class GeographyMapper:
    def __init__(self, subregion_config_path: Optional[str | Path] = None):
        self.subregion_map: Dict[str, Dict[str, str]] = {}

        if subregion_config_path:
            self.subregion_map = self._load_subregion_map(Path(subregion_config_path))

    @staticmethod
    def _load_subregion_map(path: Path) -> Dict[str, Dict[str, str]]:
        if not path.exists():
            return {}

        payload = yaml.safe_load(path.read_text()) or {}
        subregions = payload.get("subregions", {})
        mapping: Dict[str, Dict[str, str]] = {}

        for continent, region_dict in subregions.items():
            mapping[continent] = {}
            if not isinstance(region_dict, dict):
                continue

            for subregion_name, iso_list in region_dict.items():
                for iso in iso_list or []:
                    mapping[continent][iso.upper()] = subregion_name

        return mapping

    def add_geography(self, df: pd.DataFrame, iso_col: str = "iso") -> pd.DataFrame:
        """
        Adds continent and subregion columns while preserving all existing columns.
        """
        out = df.copy()

        if iso_col not in out.columns:
            raise KeyError(f"'{iso_col}' column not found in dataframe")

        out[iso_col] = out[iso_col].astype(str).str.upper()

        if "continent" not in out.columns:
            out["continent"] = out[iso_col].map(iso3_to_continent)

        def map_subregion(row):
            continent = row.get("continent", "Unknown")
            iso = str(row[iso_col]).upper()
            return self.subregion_map.get(continent, {}).get(iso, "Unassigned")

        out["subregion"] = out.apply(map_subregion, axis=1)

        return out

    
    # 1️⃣ Create a mapper instance (without subregion config for now)
mapper = GeographyMapper()

# 2️⃣ Add continent and subregion columns
iso_dataset_items_mapped = mapper.add_geography(iso_dataset_items)

# 3️⃣ Inspect the first 5 rows
iso_dataset_items_mapped.head()

,name,iso,continent,subregion
0,Afghanistan,AFG,Asia,Unassigned
1,Akrotiri and Dhekelia,XAD,Unknown,Unassigned
2,Åland,ALA,Europe,Unassigned
3,Albania,ALB,Europe,Unassigned
4,Algeria,DZA,Africa,Unassigned


In [4]:
# -----------------------------
# Continent mapping
# -----------------------------
CONTINENT_CODE_TO_NAME = {
    "AF": "Africa",
    "AS": "Asia",
    "EU": "Europe",
    "NA": "North America",
    "SA": "South America",
    "OC": "Oceania",
    "AN": "Antarctica",
}

SPECIAL_ISO3_TO_ISO2 = {"XAD", "XCA", "XCL", "XKO", "XPI", "XSP", "XSX"}


def iso3_to_continent(iso3: str) -> str:
    """Convert ISO3 → continent name."""
    if not iso3:
        return "Unknown"

    iso3 = iso3.upper().strip()

    if iso3 in SPECIAL_ISO3_TO_ISO2:
        return "Unknown"

    try:
        iso2 = pc.country_alpha3_to_country_alpha2(iso3)
        continent_code = pc.country_alpha2_to_continent_code(iso2)
        return CONTINENT_CODE_TO_NAME.get(continent_code, "Unknown")
    except Exception:
        return "Unknown"


# -----------------------------
# Geography Mapper
# -----------------------------

# ---------------------------------------------------------------------
# ISO3 -> Continent mapping
# ---------------------------------------------------------------------
# NOTE:
# This should already exist in your notebook/project. If you already have
# an `iso3_to_continent` dictionary defined elsewhere, keep that version
# and remove this placeholder.
#
# Example:
# iso3_to_continent = {
#     "KEN": "Africa",
#     "UGA": "Africa",
#     "TZA": "Africa",
#     ...
# }
# ---------------------------------------------------------------------
iso3_to_continent = globals().get("iso3_to_continent", {})


class GeographyMapper:
    """
    Adds continent and subregion metadata to country-level dataframes.

    Parameters
    ----------
    subregion_config_path : Optional[str | Path]
        Path to a YAML file containing custom subregion mappings.

    Expected YAML structure
    -----------------------
    subregions:
      Africa:
        Eastern Africa: [KEN, UGA, TZA, RWA, BDI, ETH, SOM, SSD, ERI, DJI, COM]
        Western Africa: [NGA, GHA, CIV]
      Europe:
        Northern Europe: [SWE, NOR, FIN]
    """

    def __init__(self, subregion_config_path: Optional[str | Path] = None):
        self.subregion_map: Dict[str, Dict[str, str]] = {}

        if subregion_config_path:
            self.subregion_map = self._load_subregion_map(Path(subregion_config_path))

    @staticmethod
    def _load_subregion_map(path: Path) -> Dict[str, Dict[str, str]]:
        """
        Load a YAML-based continent -> iso -> subregion mapping.

        Returns
        -------
        Dict[str, Dict[str, str]]
            Example:
            {
                "Africa": {
                    "KEN": "Eastern Africa",
                    "UGA": "Eastern Africa"
                }
            }
        """
        if not path.exists():
            print(f"[GeographyMapper] Warning: subregion config not found at {path}. Using empty mapping.")
            return {}

        payload = yaml.safe_load(path.read_text()) or {}
        subregions = payload.get("subregions", {})
        mapping: Dict[str, Dict[str, str]] = {}

        for continent, region_dict in subregions.items():
            mapping[continent] = {}

            if not isinstance(region_dict, dict):
                continue

            for subregion_name, iso_list in region_dict.items():
                for iso in iso_list or []:
                    mapping[continent][str(iso).upper()] = subregion_name

        return mapping

    def add_geography(self, df: pd.DataFrame, iso_col: str = "iso") -> pd.DataFrame:
        """
        Add continent and subregion columns to a dataframe.

        Important
        ---------
        This method preserves all existing columns in the dataframe.

        Parameters
        ----------
        df : pd.DataFrame
            Input dataframe containing an ISO3 country code column.
        iso_col : str, default="iso"
            Name of the ISO3 column.

        Returns
        -------
        pd.DataFrame
            Original dataframe plus:
            - continent
            - subregion
        """
        out = df.copy()

        if iso_col not in out.columns:
            raise KeyError(
                f"'{iso_col}' column not found in dataframe. "
                f"Available columns: {list(out.columns)}"
            )

        out[iso_col] = out[iso_col].astype(str).str.upper().str.strip()

        # Add continent if it is missing
        if "continent" not in out.columns:
            out["continent"] = out[iso_col].map(iso3_to_continent).fillna("Unknown")

        # Add subregion from YAML map if available
        def map_subregion(row) -> str:
            continent = row.get("continent", "Unknown")
            iso = str(row[iso_col]).upper()
            return self.subregion_map.get(continent, {}).get(iso, "Unassigned")

        if "subregion" not in out.columns:
            out["subregion"] = out.apply(map_subregion, axis=1)
        else:
            out["subregion"] = out["subregion"].fillna(
                out.apply(map_subregion, axis=1)
            )

        return out
# -----------------------------
# Hashing and logging utilities
# -----------------------------

def hash_dataframe(df: pd.DataFrame) -> str:
    """Create deterministic hash of dataframe."""
    df_sorted = df.sort_values(list(df.columns)).reset_index(drop=True)
    return hashlib.sha256(df_sorted.to_csv(index=False).encode()).hexdigest()


def safe_write_csv(df: pd.DataFrame, path: Path):
    """Atomic CSV write to prevent corruption."""
    with tempfile.NamedTemporaryFile(delete=False, suffix=".csv") as tmp:
        tmp_path = Path(tmp.name)

    df.to_csv(tmp_path, index=False)
    shutil.move(tmp_path, path)


def log_dataset_version(log_file: Path, record: dict):
    """Append dataset update record."""
    try:
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")
    except Exception as e:
        print(f"⚠️ logging failed: {e}")


def save_country_geography(df: pd.DataFrame):

    # Ensure schema compatibility with pipeline
    required_cols = ["name", "iso", "continent", "subregion"]
    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df[required_cols].copy()

    folder = Path("dataset")
    file_path = folder / "country_geography.csv"
    hash_file = folder / "country_geography.hash"
    log_file = folder / "dataset_versions.log"

    try:

        # -----------------------------
        # Folder creation
        # -----------------------------
        if not folder.exists():
            folder.mkdir(parents=True, exist_ok=True)

            df_hash = hash_dataframe(df)

            safe_write_csv(df, file_path)
            hash_file.write_text(df_hash)

            log_dataset_version(
                log_file,
                {
                    "dataset": "country_geography",
                    "timestamp": datetime.utcnow().isoformat(),
                    "rows": len(df),
                    "hash": df_hash,
                    "action": "created"
                },
            )

            print('✅ new folder created, "dataset" file "country_geography.csv" saved successfully')
            return

        # -----------------------------
        # File does not exist
        # -----------------------------
        if not file_path.exists():

            df_hash = hash_dataframe(df)

            safe_write_csv(df, file_path)
            hash_file.write_text(df_hash)

            log_dataset_version(
                log_file,
                {
                    "dataset": "country_geography",
                    "timestamp": datetime.utcnow().isoformat(),
                    "rows": len(df),
                    "hash": df_hash,
                    "action": "created"
                },
            )

            print('✅ file "country_geography.csv" saved in dataset successfully')
            return

        # -----------------------------
        # Compare hashes
        # -----------------------------
        new_hash = hash_dataframe(df)
        old_hash = hash_file.read_text().strip() if hash_file.exists() else None

        if new_hash == old_hash:
            print("✅ file already saved")
            return

        # -----------------------------
        # Update dataset
        # -----------------------------
        safe_write_csv(df, file_path)
        hash_file.write_text(new_hash)

        log_dataset_version(
            log_file,
            {
                "dataset": "country_geography",
                "timestamp": datetime.utcnow().isoformat(),
                "rows": len(df),
                "hash": new_hash,
                "action": "updated"
            },
        )

        print('✅ file updated and saved in dataset successfully')

    except Exception as e:
        print(f"❌ dataset save failed: {e}")
        
# -----------------------------
# Save the resulting DataFrame to CSV
# -----------------------------
mapper = GeographyMapper("subregions.yaml")

geo_df = mapper.add_geography(iso_dataset_items)

save_country_geography(geo_df)


# -----------------------------
# Run
# -----------------------------
mapper = GeographyMapper("subregions.yaml")

geo_df = mapper.add_geography(iso_dataset_items)

# Check the first 5 rows of the resulting DataFrame
print("First 5 rows of the geographic mapping:")
print(geo_df.head())

# Check the last 5 rows of the resulting DataFrame
print("Last 5 rows of the geographic mapping:")
geo_df.tail()



✅ file already saved
First 5 rows of the geographic mapping:
                    name  iso continent        subregion
0            Afghanistan  AFG      Asia    Southern Asia
1  Akrotiri and Dhekelia  XAD   Unknown       Unassigned
2                  Åland  ALA    Europe  Northern Europe
3                Albania  ALB    Europe  Southern Europe
4                Algeria  DZA    Africa  Northern Africa
Last 5 rows of the geographic mapping:


,name,iso,continent,subregion
242,"Virgin Islands, U.S.",VIR,North America,Caribbean
243,Wallis and Futuna,WLF,Oceania,Polynesia
244,Yemen,YEM,Asia,Western Asia
245,Zambia,ZMB,Africa,Eastern Africa
246,Zimbabwe,ZWE,Africa,Eastern Africa


In [5]:
# -----------------------------
# EDA 
# -----------------------------

# Check the columns of the resulting DataFrame
# print("Columns in the geographic mapping DataFrame:")
# print(geo_df.columns())

# Filter rows with unknown mappings
print("Rows with unknown continent mapping:")
geo_df[geo_df['continent'] == "Unknown"]

print("Rows with unknown subregion mapping:")
geo_df[geo_df['subregion'] == "Unassigned"]

Rows with unknown continent mapping:
Rows with unknown subregion mapping:


,name,iso,continent,subregion
1,Akrotiri and Dhekelia,XAD,Unknown,Unassigned
9,Antarctica,ATA,Unknown,Unassigned
28,"Bonaire, Sint Eustatius and Saba",BES,North America,Unassigned
33,British Indian Ocean Territory,IOT,Asia,Unassigned
48,Christmas Island,CXR,Asia,Unassigned
49,Clipperton Island,XCL,Unknown,Unassigned
50,Cocos Islands,CCK,Asia,Unassigned
80,French Southern Territories,ATF,Unknown,Unassigned
116,Kosovo,XKO,Unknown,Unassigned
173,Pitcairn Islands,PCN,Unknown,Unassigned


# Step 2: Data Input/Output (I/O) 
Data Pipeline

The data is structured across multiple sub-directories.

In [6]:
def _read_csv(path: Path) -> pd.DataFrame:
    """Helper to read CSV files with error handling."""
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path)

def load_raw_tables(data_root: str | Path) -> dict[str, pd.DataFrame]:
    """Loads all required forest data tables into a dictionary of DataFrames."""
    data_root = Path(data_root)
    
    # Check for deforestation rank in different possible directories
    def_path = data_root / "LP" / "fao_treecover_deforestation_rank.csv"
    if not def_path.exists():
        def_path = data_root / "MP" / "fao_treecover_deforestation_rank.csv"

    return {
        "iso_meta": _read_csv(data_root /   "country_geography.csv"),
        "extent": _read_csv(data_root / "HP" / "treecover_extent_2000_by_region__ha.csv"),
        "gain": _read_csv(data_root / "HP" / "treecover_gain_2000-2020_by_region__ha.csv"),
        "net_change": _read_csv(data_root / "HP" / "net_tree_cover_change_from_height__ha.csv"),
        "loss_annual": _read_csv(data_root / "HP" / "treecover_loss_by_region__ha.csv"),
        "primary_loss_annual": _read_csv(data_root / "HP" / "treecover_loss_in_primary_forests_2001_tropics_only__ha.csv"),
        "deforestation_rank": _read_csv(def_path),
        "reforestation": _read_csv(data_root / "MP" / "fao_treecover_reforestation__ha.csv"),
        "driver_global": _read_csv(data_root / "MP" / "tree_cover_loss_by_driver.csv"),
    }


# -----------------------------
# Quick check of loaded tables
# -----------------------------

tables = load_raw_tables("dataset")


for name, df in tables.items():
    print(f"\n{name}")
    print("-" * 40)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    print(df.head(3))


iso_meta
----------------------------------------
Shape: (247, 4)
Columns: ['name', 'iso', 'continent', 'subregion']
                    name  iso continent        subregion
0            Afghanistan  AFG      Asia    Southern Asia
1  Akrotiri and Dhekelia  XAD   Unknown       Unassigned
2                  Åland  ALA    Europe  Northern Europe

extent
----------------------------------------
Shape: (235, 3)
Columns: ['iso', 'umd_tree_cover_extent_2000__ha', 'area__ha']
   iso  umd_tree_cover_extent_2000__ha      area__ha
0  ABW                    2.495241e+01  1.819751e+04
1  AFG                    2.057711e+05  6.438365e+07
2  AGO                    5.527613e+07  1.246914e+08

gain
----------------------------------------
Shape: (219, 2)
Columns: ['iso', 'umd_tree_cover_gain__ha']
   iso  umd_tree_cover_gain__ha
0  ABW             1.923964e+01
1  AFG             1.073873e+04
2  AGO             1.224132e+06

net_change
----------------------------------------
Shape: (256, 8)
Columns: [

In [7]:
class ForestFeatureBuilder:
    """
    Build country-level and country-year forest analytics tables.

    Parameters
    ----------
    geography_mapper : GeographyMapper
        Mapper used to add continent and subregion labels.
    """

    def __init__(self, geography_mapper: GeographyMapper):
        self.geography_mapper = geography_mapper

    # -----------------------------------------------------------------
    # Internal utilities
    # -----------------------------------------------------------------
    @staticmethod
    def _require_tables(tables: Dict[str, pd.DataFrame], required_tables: List[str]) -> None:
        """
        Ensure required tables exist in the input dictionary.
        """
        missing = [t for t in required_tables if t not in tables]
        if missing:
            raise KeyError(
                f"Missing required table(s): {missing}. "
                f"Available tables: {list(tables.keys())}"
            )

    @staticmethod
    def _require_columns(df: pd.DataFrame, required_cols: List[str], table_name: str) -> None:
        """
        Ensure required columns exist in a dataframe.
        """
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise KeyError(
                f"Missing required column(s) {missing} in table '{table_name}'. "
                f"Available columns: {list(df.columns)}"
            )

    @staticmethod
    def _safe_rename(df: pd.DataFrame, rename_map: Dict[str, str]) -> pd.DataFrame:
        """
        Rename only columns that actually exist.
        """
        applicable = {k: v for k, v in rename_map.items() if k in df.columns}
        return df.rename(columns=applicable)

    @staticmethod
    def _ensure_columns(df: pd.DataFrame, columns_with_defaults: Dict[str, object]) -> pd.DataFrame:
        """
        Ensure specific columns exist; add them with default values if missing.
        """
        out = df.copy()
        for col, default in columns_with_defaults.items():
            if col not in out.columns:
                out[col] = default
        return out

    # -----------------------------------------------------------------
    # Canonical country table
    # -----------------------------------------------------------------
    def build_canonical_country_table(self, tables: Dict[str, pd.DataFrame]) -> pd.DataFrame:
        """
        Build a canonical country-level table.

        Expected output
        ---------------
        One row per country with:
        - iso
        - country_name
        - continent
        - subregion
        - area_ha
        - extent_2000_ha
        - gain_2000_2020_ha
        - loss_total_2001_2020
        - primary_loss_total_2001_2020
        - avg_annual_loss_ha
        - loss_per_area
        - net_change_ha
        - net_change_pct
        - recovery_gap_ha
        - reforestation_rate
        - deforestation_rank_value
        - disturbance_ha
        """
        self._require_tables(
            tables,
            required_tables=["iso_meta", "extent", "gain", "loss_annual", "primary_loss_annual"]
        )

        # -----------------------------
        # 1. ISO metadata
        # -----------------------------
        iso_meta = tables["iso_meta"].copy()

        # Standardize common name column to country_name
        if "name" in iso_meta.columns and "country_name" not in iso_meta.columns:
            iso_meta = iso_meta.rename(columns={"name": "country_name"})

        self._require_columns(iso_meta, ["iso"], "iso_meta")

        # If country_name is absent, create a placeholder from iso
        if "country_name" not in iso_meta.columns:
            iso_meta["country_name"] = iso_meta["iso"]

        iso_meta["iso"] = iso_meta["iso"].astype(str).str.upper().str.strip()

        # -----------------------------
        # 2. Extent table
        # -----------------------------
        extent = tables["extent"].copy()
        self._require_columns(extent, ["iso"], "extent")
        extent["iso"] = extent["iso"].astype(str).str.upper().str.strip()

        extent = self._safe_rename(
            extent,
            {
                # common variants
                "umd_tree_cover_extent_2000__ha": "extent_2000_ha",
                "extent_2000": "extent_2000_ha",
                "tree_cover_extent_2000_ha": "extent_2000_ha",
                "area": "area_ha",
                "country_area_ha": "area_ha",
            }
        )

        # -----------------------------
        # 3. Gain table
        # -----------------------------
        gain = tables["gain"].copy()
        self._require_columns(gain, ["iso"], "gain")
        gain["iso"] = gain["iso"].astype(str).str.upper().str.strip()

        gain = self._safe_rename(
            gain,
            {
                "umd_tree_cover_gain_2000_2020__ha": "gain_2000_2020_ha",
                "gain_2000_2020": "gain_2000_2020_ha",
                "tree_cover_gain_2000_2020_ha": "gain_2000_2020_ha",
            }
        )

        # -----------------------------
        # 4. Aggregate annual loss table
        # -----------------------------
        loss_annual = tables["loss_annual"].copy()
        self._require_columns(loss_annual, ["iso"], "loss_annual")
        loss_annual["iso"] = loss_annual["iso"].astype(str).str.upper().str.strip()

        loss_annual = self._safe_rename(
            loss_annual,
            {
                "umd_tree_cover_loss__year": "year",
                "umd_tree_cover_loss__ha": "annual_loss_ha",
                "gfw_gross_emissions_co2e_all_gases__Mg": "annual_emissions_Mg",
            }
        )

        self._require_columns(loss_annual, ["iso", "year", "annual_loss_ha"], "loss_annual")

        loss_agg = (
            loss_annual.groupby("iso", as_index=False)
            .agg(
                loss_total_2001_2020=("annual_loss_ha", "sum"),
                avg_annual_loss_ha=("annual_loss_ha", "mean"),
            )
        )

        # -----------------------------
        # 5. Aggregate annual primary forest loss table
        # -----------------------------
        primary_loss_annual = tables["primary_loss_annual"].copy()
        self._require_columns(primary_loss_annual, ["iso"], "primary_loss_annual")
        primary_loss_annual["iso"] = primary_loss_annual["iso"].astype(str).str.upper().str.strip()

        primary_loss_annual = self._safe_rename(
            primary_loss_annual,
            {
                "umd_tree_cover_loss__year": "year",
                "umd_tree_cover_loss__ha": "annual_primary_loss_ha",
                "gfw_gross_emissions_co2e_all_gases__Mg": "annual_primary_emissions_Mg",
            }
        )

        self._require_columns(
            primary_loss_annual,
            ["iso", "year", "annual_primary_loss_ha"],
            "primary_loss_annual"
        )

        primary_loss_agg = (
            primary_loss_annual.groupby("iso", as_index=False)
            .agg(
                primary_loss_total_2001_2020=("annual_primary_loss_ha", "sum"),
            )
        )

        # -----------------------------
        # 6. Merge all static country components
        # -----------------------------
        canonical = (
            iso_meta
            .merge(extent.drop(columns=["name"], errors="ignore"), on="iso", how="left")
            .merge(gain.drop(columns=["name"], errors="ignore"), on="iso", how="left")
            .merge(loss_agg, on="iso", how="left")
            .merge(primary_loss_agg, on="iso", how="left")
        )

        # Add geography labels while preserving existing columns
        canonical = self.geography_mapper.add_geography(canonical)

        # -----------------------------
        # 7. Ensure expected base columns exist
        # -----------------------------
        canonical = self._ensure_columns(
            canonical,
            {
                "country_name": pd.NA,
                "area_ha": pd.NA,
                "extent_2000_ha": pd.NA,
                "gain_2000_2020_ha": 0.0,
                "loss_total_2001_2020": 0.0,
                "primary_loss_total_2001_2020": 0.0,
                "avg_annual_loss_ha": pd.NA,
            }
        )

        # -----------------------------
        # 8. Derived fields
        # -----------------------------
        canonical["gain_2000_2020_ha"] = pd.to_numeric(
            canonical["gain_2000_2020_ha"], errors="coerce"
        )
        canonical["loss_total_2001_2020"] = pd.to_numeric(
            canonical["loss_total_2001_2020"], errors="coerce"
        )
        canonical["primary_loss_total_2001_2020"] = pd.to_numeric(
            canonical["primary_loss_total_2001_2020"], errors="coerce"
        )
        canonical["extent_2000_ha"] = pd.to_numeric(
            canonical["extent_2000_ha"], errors="coerce"
        )
        canonical["area_ha"] = pd.to_numeric(
            canonical["area_ha"], errors="coerce"
        )
        canonical["avg_annual_loss_ha"] = pd.to_numeric(
            canonical["avg_annual_loss_ha"], errors="coerce"
        )

        # Net change = gain - loss
        canonical["net_change_ha"] = (
            canonical["gain_2000_2020_ha"].fillna(0) -
            canonical["loss_total_2001_2020"].fillna(0)
        )

        # Net change as % of extent_2000_ha
        canonical["net_change_pct"] = (
            canonical["net_change_ha"] /
            canonical["extent_2000_ha"].replace(0, np.nan)
        ) * 100

        # Recovery gap = gain - loss
        canonical["recovery_gap_ha"] = (
            canonical["gain_2000_2020_ha"].fillna(0) -
            canonical["loss_total_2001_2020"].fillna(0)
        )

        # Reforestation rate = gain / loss
        canonical["reforestation_rate"] = (
            canonical["gain_2000_2020_ha"] /
            canonical["loss_total_2001_2020"].replace(0, np.nan)
        )

        # Loss per area
        canonical["loss_per_area"] = (
            canonical["loss_total_2001_2020"] /
            canonical["area_ha"].replace(0, np.nan)
        )

        # Placeholder for disturbance until a disturbance source table is added
        if "disturbance_ha" not in canonical.columns:
            canonical["disturbance_ha"] = pd.NA

        # Rank countries by cumulative forest loss
        canonical["deforestation_rank_value"] = canonical["loss_total_2001_2020"].rank(
            method="dense",
            ascending=False
        )

        # Remove possible duplicates if source tables were not fully unique
        canonical = canonical.drop_duplicates(subset=["iso"]).reset_index(drop=True)

        return canonical

    # -----------------------------------------------------------------
    # Country-year feature store
    # -----------------------------------------------------------------
    def build_country_year_feature_store(self, tables: Dict[str, pd.DataFrame]) -> pd.DataFrame:
        """
        Build the country-year feature store.

        Output
        ------
        One row per (iso, year) containing:
        - annual loss features
        - annual primary loss features
        - annual emissions
        - static country metadata
        - lag features
        - rolling features
        """
        self._require_tables(
            tables,
            required_tables=["loss_annual", "primary_loss_annual", "iso_meta", "extent", "gain"]
        )

        # -----------------------------
        # 1. Standardize annual loss table
        # -----------------------------
        loss = tables["loss_annual"].copy()
        loss = self._safe_rename(
            loss,
            {
                "umd_tree_cover_loss__year": "year",
                "umd_tree_cover_loss__ha": "annual_loss_ha",
                "gfw_gross_emissions_co2e_all_gases__Mg": "annual_emissions_Mg",
            }
        )

        self._require_columns(loss, ["iso", "year", "annual_loss_ha"], "loss_annual")

        loss["iso"] = loss["iso"].astype(str).str.upper().str.strip()
        loss["year"] = pd.to_numeric(loss["year"], errors="coerce")

        if "annual_emissions_Mg" not in loss.columns:
            loss["annual_emissions_Mg"] = pd.NA

        # -----------------------------
        # 2. Standardize annual primary loss table
        # -----------------------------
        primary = tables["primary_loss_annual"].copy()
        primary = self._safe_rename(
            primary,
            {
                "umd_tree_cover_loss__year": "year",
                "umd_tree_cover_loss__ha": "annual_primary_loss_ha",
                "gfw_gross_emissions_co2e_all_gases__Mg": "annual_primary_emissions_Mg",
            }
        )

        self._require_columns(primary, ["iso", "year", "annual_primary_loss_ha"], "primary_loss_annual")

        primary["iso"] = primary["iso"].astype(str).str.upper().str.strip()
        primary["year"] = pd.to_numeric(primary["year"], errors="coerce")

        if "annual_primary_emissions_Mg" not in primary.columns:
            primary["annual_primary_emissions_Mg"] = pd.NA

        # -----------------------------
        # 3. Build static canonical country table
        # -----------------------------
        country = self.build_canonical_country_table(tables)

        # -----------------------------
        # 4. Merge annual tables
        # -----------------------------
        out = loss.merge(
            primary,
            on=["iso", "year"],
            how="left",
            suffixes=("", "_primary_dup")
        )

        # -----------------------------
        # 5. Join static country metadata
        # -----------------------------
        static_cols = [
            "iso",
            "country_name",
            "continent",
            "subregion",
            "area_ha",
            "extent_2000_ha",
            "gain_2000_2020_ha",
            "loss_total_2001_2020",
            "primary_loss_total_2001_2020",
            "avg_annual_loss_ha",
            "loss_per_area",
            "net_change_ha",
            "net_change_pct",
            "recovery_gap_ha",
            "disturbance_ha",
            "reforestation_rate",
            "deforestation_rank_value",
        ]

        available_static_cols = [c for c in static_cols if c in country.columns]
        missing_static_cols = [c for c in static_cols if c not in country.columns]

        if missing_static_cols:
            print("[ForestFeatureBuilder] Missing static columns in canonical country table:")
            print(missing_static_cols)

        out = (
            out.merge(country[available_static_cols], on="iso", how="left")
               .sort_values(["iso", "year"])
               .reset_index(drop=True)
        )

        # -----------------------------
        # 6. Numeric cleanup
        # -----------------------------
        numeric_cols = [
            "annual_loss_ha",
            "annual_emissions_Mg",
            "annual_primary_loss_ha",
            "annual_primary_emissions_Mg",
        ]

        for col in numeric_cols:
            if col in out.columns:
                out[col] = pd.to_numeric(out[col], errors="coerce")

        # -----------------------------
        # 7. Grouped modelling features
        # -----------------------------
        grp = out.groupby("iso", group_keys=False)

        # Lag features
        out["annual_loss_ha_lag1"] = grp["annual_loss_ha"].shift(1)
        out["annual_loss_ha_lag2"] = grp["annual_loss_ha"].shift(2)
        out["annual_primary_loss_ha_lag1"] = grp["annual_primary_loss_ha"].shift(1)
        out["annual_emissions_Mg_lag1"] = grp["annual_emissions_Mg"].shift(1)

        # Rolling averages
        out["annual_loss_ha_roll3"] = grp["annual_loss_ha"].transform(
            lambda s: s.rolling(window=3, min_periods=1).mean()
        )
        out["annual_loss_ha_roll5"] = grp["annual_loss_ha"].transform(
            lambda s: s.rolling(window=5, min_periods=1).mean()
        )
        out["annual_primary_loss_ha_roll3"] = grp["annual_primary_loss_ha"].transform(
            lambda s: s.rolling(window=3, min_periods=1).mean()
        )
        out["annual_emissions_Mg_roll3"] = grp["annual_emissions_Mg"].transform(
            lambda s: s.rolling(window=3, min_periods=1).mean()
        )

        # Year-over-year change
        out["annual_loss_ha_yoy_change"] = grp["annual_loss_ha"].diff()
        out["annual_primary_loss_ha_yoy_change"] = grp["annual_primary_loss_ha"].diff()

        # Growth rates
        out["annual_loss_ha_growth_rate"] = (
            grp["annual_loss_ha"].pct_change()
        )
        out["annual_primary_loss_ha_growth_rate"] = (
            grp["annual_primary_loss_ha"].pct_change()
        )

        # Loss relative to historical country extent
        if "extent_2000_ha" in out.columns:
            out["annual_loss_pct_of_extent_2000"] = (
                out["annual_loss_ha"] / out["extent_2000_ha"].replace(0, np.nan)
            ) * 100

            out["annual_primary_loss_pct_of_extent_2000"] = (
                out["annual_primary_loss_ha"] / out["extent_2000_ha"].replace(0, np.nan)
            ) * 100

        return out

In [8]:
# Initialize GeographyMapper
geo_mapper = GeographyMapper(subregion_config_path=None)

# Initialize ForestFeatureBuilder
ffb = ForestFeatureBuilder(geography_mapper=geo_mapper)

# Build canonical country table
canonical_country_table = ffb.build_canonical_country_table(tables)

# Build country-year feature store
country_year_features = ffb.build_country_year_feature_store(tables)

print("Canonical country table shape:", canonical_country_table.shape)
print("Country-year feature store shape:", country_year_features.shape)

Canonical country table shape: (247, 19)
Country-year feature store shape: (4776, 36)


In [9]:
# ---------------------------------------------------------
# Validation: inspect canonical table columns
# ---------------------------------------------------------
print("Canonical country table columns:")
print(canonical_country_table.columns.tolist())

print("\nSample canonical country table:")
display(canonical_country_table.head())

# ---------------------------------------------------------
# Validation: inspect country-year feature store columns
# ---------------------------------------------------------
print("\nCountry-year feature store columns:")
print(country_year_features.columns.tolist())

print("\nSample country-year feature store:")
display(country_year_features.head())

# ---------------------------------------------------------
# Validation: missing value summary for critical columns
# ---------------------------------------------------------
critical_cols = [
    "iso",
    "country_name",
    "continent",
    "subregion",
    "annual_loss_ha",
    "annual_primary_loss_ha",
]

existing_critical_cols = [c for c in critical_cols if c in country_year_features.columns]

print("\nMissing values in critical columns:")
print(country_year_features[existing_critical_cols].isna().sum())

Canonical country table columns:
['country_name', 'iso', 'continent', 'subregion', 'extent_2000_ha', 'area__ha', 'umd_tree_cover_gain__ha', 'loss_total_2001_2020', 'avg_annual_loss_ha', 'primary_loss_total_2001_2020', 'area_ha', 'gain_2000_2020_ha', 'net_change_ha', 'net_change_pct', 'recovery_gap_ha', 'reforestation_rate', 'loss_per_area', 'disturbance_ha', 'deforestation_rank_value']

Sample canonical country table:


,country_name,iso,continent,subregion,extent_2000_ha,area__ha,umd_tree_cover_gain__ha,loss_total_2001_2020,avg_annual_loss_ha,primary_loss_total_2001_2020,area_ha,gain_2000_2020_ha,net_change_ha,net_change_pct,recovery_gap_ha,reforestation_rate,loss_per_area,disturbance_ha,deforestation_rank_value
0,Afghanistan,AFG,Asia,Southern Asia,2.057711e+05,6.438365e+07,10738.734394,1912.066093,91.050766,NaN,NaN,0.0,-1912.066093,-0.929220,-1912.066093,0.0,NaN,<NA>,156.0
1,Akrotiri and Dhekelia,XAD,Unknown,Unassigned,9.112420e+02,4.673448e+04,83.872300,76.132899,3.460586,NaN,NaN,0.0,-76.132899,-8.354850,-76.132899,0.0,NaN,<NA>,188.0
2,Åland,ALA,Europe,Northern Europe,1.077392e+05,1.506139e+05,2582.869603,17655.780082,735.657503,NaN,NaN,0.0,-17655.780082,-16.387512,-17655.780082,0.0,NaN,<NA>,131.0
3,Albania,ALB,Europe,Southern Europe,6.484594e+05,2.872761e+06,16468.972163,47318.591026,1971.607959,NaN,NaN,0.0,-47318.591026,-7.297079,-47318.591026,0.0,NaN,<NA>,116.0
4,Algeria,DZA,Africa,Northern Africa,1.223324e+06,2.308021e+08,89145.649956,231653.346546,9652.222773,NaN,NaN,0.0,-231653.346546,-18.936383,-231653.346546,0.0,NaN,<NA>,89.0



Country-year feature store columns:
['iso', 'year', 'annual_loss_ha', 'annual_emissions_Mg', 'annual_primary_loss_ha', 'annual_primary_emissions_Mg', 'country_name', 'continent', 'subregion', 'area_ha', 'extent_2000_ha', 'gain_2000_2020_ha', 'loss_total_2001_2020', 'primary_loss_total_2001_2020', 'avg_annual_loss_ha', 'loss_per_area', 'net_change_ha', 'net_change_pct', 'recovery_gap_ha', 'disturbance_ha', 'reforestation_rate', 'deforestation_rank_value', 'annual_loss_ha_lag1', 'annual_loss_ha_lag2', 'annual_primary_loss_ha_lag1', 'annual_emissions_Mg_lag1', 'annual_loss_ha_roll3', 'annual_loss_ha_roll5', 'annual_primary_loss_ha_roll3', 'annual_emissions_Mg_roll3', 'annual_loss_ha_yoy_change', 'annual_primary_loss_ha_yoy_change', 'annual_loss_ha_growth_rate', 'annual_primary_loss_ha_growth_rate', 'annual_loss_pct_of_extent_2000', 'annual_primary_loss_pct_of_extent_2000']

Sample country-year feature store:


,iso,year,annual_loss_ha,annual_emissions_Mg,annual_primary_loss_ha,annual_primary_emissions_Mg,country_name,continent,subregion,area_ha,...,annual_loss_ha_roll3,annual_loss_ha_roll5,annual_primary_loss_ha_roll3,annual_emissions_Mg_roll3,annual_loss_ha_yoy_change,annual_primary_loss_ha_yoy_change,annual_loss_ha_growth_rate,annual_primary_loss_ha_growth_rate,annual_loss_pct_of_extent_2000,annual_primary_loss_pct_of_extent_2000
0,ABW,2002,0.526108,289.581373,NaN,NaN,Aruba,North America,Caribbean,NaN,...,0.526108,0.526108,NaN,289.581373,NaN,NaN,NaN,NaN,2.108444,NaN
1,ABW,2003,0.977058,550.205498,NaN,NaN,Aruba,North America,Caribbean,NaN,...,0.751583,0.751583,NaN,419.893435,0.450950,NaN,0.857144,NaN,3.915684,NaN
2,ABW,2006,0.225487,44.168715,0.075162,24.575414,Aruba,North America,Caribbean,NaN,...,0.576217,0.576217,0.075162,294.651862,-0.751571,NaN,-0.769218,NaN,0.903668,0.301223
3,ABW,2009,0.375798,105.367894,0.300651,85.674361,Aruba,North America,Caribbean,NaN,...,0.526114,0.526113,0.187907,233.247369,0.150311,0.225489,0.666606,3.000018,1.506058,1.204897
4,ABW,2012,0.150325,26.254148,NaN,NaN,Aruba,North America,Caribbean,NaN,...,0.250537,0.450955,0.187907,58.596919,-0.225473,NaN,-0.599985,NaN,0.602446,NaN



Missing values in critical columns:
iso                          0
country_name               122
continent                  122
subregion                  122
annual_loss_ha               0
annual_primary_loss_ha    2551
dtype: int64


In [10]:
country = ffb.build_canonical_country_table(tables)
print(country.columns.tolist())

['country_name', 'iso', 'continent', 'subregion', 'extent_2000_ha', 'area__ha', 'umd_tree_cover_gain__ha', 'loss_total_2001_2020', 'avg_annual_loss_ha', 'primary_loss_total_2001_2020', 'area_ha', 'gain_2000_2020_ha', 'net_change_ha', 'net_change_pct', 'recovery_gap_ha', 'reforestation_rate', 'loss_per_area', 'disturbance_ha', 'deforestation_rank_value']


In [13]:
for name, df in tables.items():
    print(f"\n{name}")
    print(df.columns.tolist())


iso_meta
['name', 'iso', 'continent', 'subregion']

extent
['iso', 'umd_tree_cover_extent_2000__ha', 'area__ha']

gain
['iso', 'umd_tree_cover_gain__ha']

net_change
['iso', 'stable', 'loss', 'gain', 'disturb', 'net', 'change', 'gfw_area__ha']

loss_annual
['iso', 'umd_tree_cover_loss__year', 'umd_tree_cover_loss__ha', 'gfw_gross_emissions_co2e_all_gases__Mg']

primary_loss_annual
['iso', 'umd_tree_cover_loss__year', 'umd_tree_cover_loss__ha', 'gfw_gross_emissions_co2e_all_gases__Mg']

deforestation_rank
['iso', 'country', 'def_per_year']

reforestation
['iso', 'name', 'year', 'reforestation__rate']

driver_global
['drivers_type', 'loss_year', 'loss_area_ha', 'gross_carbon_emissions_Mg']
